In [1]:
import os
import re
import csv
import random
import shutil
from datetime import datetime
import torch
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import RobertaTokenizer, RobertaModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from tqdm import tqdm
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(2137)

c:\Projects\CS4928-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


How well can RoBERTa identify three Gang of Four patterns (Factory Method, Prototype, Singleton) in Java source code across 30 examples (15 true, 15 false) each?

In [9]:
# Where the frozen RQ1 samples are written (relative to this notebooks/ folder)
RQ1_ROOT = "../datasets/rq1_datasets"

# Positives are the keyword-named copies in <Pattern>/name/ (unambiguous IDs);
# negatives come from each pattern's non-* folder.
RQ1_SOURCES = {
    "factory_method": {
        "true":  "../datasets/Factory method/name",             # 11 *.java
        "false": "../datasets/Factory method/nonfactorymethod", # 82 *.java
    },
    "prototype": {
        "true":  "../datasets/Prototype/name",                  # 32 *.java
        "false": "../datasets/Prototype/nonprototype",          # 61 *.java
    },
    "singleton": {
        "true":  "../datasets/Singleton/name",                  # 25 *.java
        "false": "../datasets/Singleton/nonsingleton",          # 68 *.java
    },
}


def _natural_key(name):
    """Sort 'foo (2).java' before 'foo (10).java'."""
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", name)]


def build_rq1_dataset(pattern, n_true=15, n_false=None, seed=None):
    """Copy one frozen RQ1 sample into ../datasets/rq1_datasets/<pattern>/.

    pattern : "factory_method" | "prototype" | "singleton"
    n_true  : number of TRUE examples (capped at what the dataset provides)
    n_false : number of FALSE examples; None -> same as the actual n_true (1:1)
    seed    : None -> first N files in stable (natural) sorted order
              int  -> reproducible random sample using that seed

    Creates <pattern>/pattern/ (label 1) and <pattern>/non-pattern/ (label 0),
    rebuilds both folders on every call, and writes <pattern>/selection.csv
    with the frozen file IDs.
    """
    if pattern not in RQ1_SOURCES:
        raise ValueError(f"Unknown pattern {pattern!r}; choose from {list(RQ1_SOURCES)}")

    src_true  = RQ1_SOURCES[pattern]["true"]
    src_false = RQ1_SOURCES[pattern]["false"]

    true_files  = sorted((f for f in os.listdir(src_true)  if f.endswith(".java")), key=_natural_key)
    false_files = sorted((f for f in os.listdir(src_false) if f.endswith(".java")), key=_natural_key)

    # "max per class": never copy more than the dataset actually has
    n_true_sel  = min(n_true, len(true_files))
    n_false_sel = min(n_true_sel if n_false is None else n_false, len(false_files))

    if n_true_sel < n_true:
        print(f"  note: {pattern} has only {len(true_files)} true examples; using all of them.")
    if n_false is not None and n_false_sel < n_false:
        print(f"  note: {pattern} has only {len(false_files)} false examples; using all of them.")

    # pick the sample
    if seed is None:
        sel_true, sel_false = true_files[:n_true_sel], false_files[:n_false_sel]
        rule = "first N files in stable sorted order"
    else:
        rng = random.Random(seed)
        sel_true  = rng.sample(true_files,  n_true_sel)
        sel_false = rng.sample(false_files, n_false_sel)
        rule = f"random sample, seed={seed}"

    # rebuild the destination so it contains exactly this selection
    target    = os.path.join(RQ1_ROOT, pattern)
    dst_true  = os.path.join(target, "pattern")
    dst_false = os.path.join(target, "non-pattern")
    if os.path.exists(target):
        shutil.rmtree(target)
    os.makedirs(dst_true)
    os.makedirs(dst_false)

    for f in sel_true:
        shutil.copy2(os.path.join(src_true, f), os.path.join(dst_true, f))
    for f in sel_false:
        shutil.copy2(os.path.join(src_false, f), os.path.join(dst_false, f))

    # freeze the choice on disk (evidence for the report)
    with open(os.path.join(target, "selection.csv"), "w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["class", "label", "filename", "source_folder"])
        writer.writerows([["pattern", 1, f, src_true] for f in sel_true])
        writer.writerows([["non-pattern", 0, f, src_false] for f in sel_false])
        writer.writerow([])
        writer.writerow([f"selection rule: {rule}"])
        writer.writerow([f"created: {datetime.now().isoformat(timespec='seconds')}"])

    print(f"{pattern}: {n_true_sel} true + {n_false_sel} false -> {target}")
    print(f'  java_programs_folder = "{target}"')
    print(f'  class_folders = {{"non-pattern": 0, "pattern": 1}}')

    return {"pattern": pattern, "n_true": n_true_sel, "n_false": n_false_sel, "folder": target}


def build_all_rq1(n_true=15, n_false=None, seed=None):
    """Build all three RQ1 patterns with the same true/false settings."""
    results = [build_rq1_dataset(p, n_true=n_true, n_false=n_false, seed=seed)
               for p in RQ1_SOURCES]
    print("\nSummary")
    for r in results:
        print(f"  {r['pattern']:15s} {r['n_true']:2d} true + {r['n_false']:2d} false "
              f"({r['n_true'] + r['n_false']:2d} files) -> {r['folder']}")
    return results

In [10]:
build_all_rq1(n_true=15, n_false=None)

  note: factory_method has only 11 true examples; using all of them.
factory_method: 11 true + 11 false -> ../datasets/rq1_datasets/factory_method
  java_programs_folder = "../datasets/rq1_datasets/factory_method"
  class_folders = {"non-pattern": 0, "pattern": 1}
prototype: 15 true + 15 false -> ../datasets/rq1_datasets/prototype
  java_programs_folder = "../datasets/rq1_datasets/prototype"
  class_folders = {"non-pattern": 0, "pattern": 1}
singleton: 15 true + 15 false -> ../datasets/rq1_datasets/singleton
  java_programs_folder = "../datasets/rq1_datasets/singleton"
  class_folders = {"non-pattern": 0, "pattern": 1}

Summary
  factory_method  11 true + 11 false (22 files) -> ../datasets/rq1_datasets/factory_method
  prototype       15 true + 15 false (30 files) -> ../datasets/rq1_datasets/prototype
  singleton       15 true + 15 false (30 files) -> ../datasets/rq1_datasets/singleton


[{'pattern': 'factory_method',
  'n_true': 11,
  'n_false': 11,
  'folder': '../datasets/rq1_datasets/factory_method'},
 {'pattern': 'prototype',
  'n_true': 15,
  'n_false': 15,
  'folder': '../datasets/rq1_datasets/prototype'},
 {'pattern': 'singleton',
  'n_true': 15,
  'n_false': 15,
  'folder': '../datasets/rq1_datasets/singleton'}]

In [ ]:
# Define the path to the folder containing your Java programs
java_programs_folder = "../datasets/rq1_datasets/factory_method" 

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device)

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {"non-pattern": 0, "pattern": 1}

for folder_name, label in class_folders.items():
    folder_path = os.path.join(java_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                java_programs.append(f.read())

            program_labels.append(label)
            program_files.append(program_file)


from collections import Counter
print(Counter(program_labels))

# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# Get k-nearest neighbors indices for each program
k = 5  # Number of neighbors to consider
knn_indices = np.argsort(distance_matrix)[:, 1:k+1]

# Initialize an array to store predicted labels
predicted_labels = []

# Predict labels for each program based on the majority label of neighbors
for indices in knn_indices:
    neighbor_labels = [program_labels[i] for i in indices]
    predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
    predicted_labels.append(predicted_label)

# Convert predicted_labels to NumPy array for further analysis
predicted_labels = np.array(predicted_labels)

# Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(normalized_embeddings, predicted_labels, test_size=0.2, random_state=0)

X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    np.vstack(program_embeddings),
    program_labels,
    program_files,
    test_size=0.2,
    random_state=0
)

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Counter({0: 11, 1: 11})


Generating embeddings: 100%|██████████| 22/22 [03:38<00:00,  9.95s/it]

File: factorymethod (2).java, Predicted Label: 1, True Label: 1
File: nonfm (10).java, Predicted Label: 0, True Label: 0
File: factorymethod (10).java, Predicted Label: 1, True Label: 1
File: factorymethod (5).java, Predicted Label: 1, True Label: 1
File: nonfm (2).java, Predicted Label: 0, True Label: 0
Precision: 1.0, Recall: 1.0, F-score: 1.0


In [ ]:
# Define the path to the folder containing your Java programs
java_programs_folder = "../datasets/rq1_datasets/prototype"

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device)

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {"non-pattern": 0, "pattern": 1}

for folder_name, label in class_folders.items():
    folder_path = os.path.join(java_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                java_programs.append(f.read())

            program_labels.append(label)
            program_files.append(program_file)


from collections import Counter
print(Counter(program_labels))

# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# Get k-nearest neighbors indices for each program
k = 5  # Number of neighbors to consider
knn_indices = np.argsort(distance_matrix)[:, 1:k+1]

# Initialize an array to store predicted labels
predicted_labels = []

# Predict labels for each program based on the majority label of neighbors
for indices in knn_indices:
    neighbor_labels = [program_labels[i] for i in indices]
    predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
    predicted_labels.append(predicted_label)

# Convert predicted_labels to NumPy array for further analysis
predicted_labels = np.array(predicted_labels)

# Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(normalized_embeddings, predicted_labels, test_size=0.2, random_state=0)

X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    np.vstack(program_embeddings),
    program_labels,
    program_files,
    test_size=0.2,
    random_state=0
)

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Counter({0: 15, 1: 15})


Generating embeddings:  27%|██▋       | 8/30 [00:24<01:00,  2.77s/it]

In [ ]:
# Define the path to the folder containing your Java programs
java_programs_folder = "../datasets/rq1_datasets/singleton"

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device)

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {"non-pattern": 0, "pattern": 1}

for folder_name, label in class_folders.items():
    folder_path = os.path.join(java_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                java_programs.append(f.read())

            program_labels.append(label)
            program_files.append(program_file)


print(Counter(program_labels))

# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# Get k-nearest neighbors indices for each program
k = 5  # Number of neighbors to consider
knn_indices = np.argsort(distance_matrix)[:, 1:k+1]

# Initialize an array to store predicted labels
predicted_labels = []

# Predict labels for each program based on the majority label of neighbors
for indices in knn_indices:
    neighbor_labels = [program_labels[i] for i in indices]
    predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
    predicted_labels.append(predicted_label)

# Convert predicted_labels to NumPy array for further analysis
predicted_labels = np.array(predicted_labels)

# Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(normalized_embeddings, predicted_labels, test_size=0.2, random_state=0)

X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    normalized_embeddings,
    predicted_labels,
    program_files,
    test_size=0.2,
    random_state=0
)

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 435.63it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Counter({0: 68, 1: 25})


Generating embeddings:  73%|███████▎  | 68/93 [05:04<02:58,  7.12s/it]